# Shared Variables: Broadcast and Accumulator

---

## 6. Shared Variables: Broadcast and Accumulator

### 6.1 Broadcast Variables

Broadcast variables are read-only variables that are **cached on each machine** rather than shipped with tasks. This avoids redundant copies.

Broadcast variables are read-only shared variables that are cached on each worker node instead of being sent with every task.
They help avoid data transfer overhead when you need the same large lookup or configuration data across tasks.

Why Use Broadcast Variables?

In normal operations, Spark sends a copy of each variable to every task.

For large datasets (like lookup tables), this can cause excessive network and memory overhead.

Broadcast variables send the data only once to each executor and cache it there.

Use case: Lookups, configuration values, static reference data.

#### Example:

```

In [ ]:
from pyspark import SparkContext

sc = SparkContext("local", "BroadcastExample")

# Large lookup data (sent once to all executors)
product_info = {"p1": "TV", "p2": "Mobile", "p3": "Laptop"}

# Broadcast it
broadcast_var = sc.broadcast(product_info)

# Example RDD
transactions = sc.parallelize([("p1", 1000), ("p2", 2000), ("p3", 1500)])

# Access broadcast variable inside transformation
result = transactions.map(lambda x: (broadcast_var.value[x[0]], x[1]))

print(result.collect())

# Output [('TV', 1000), ('Mobile', 2000), ('Laptop', 1500)]


[('TV', 1000), ('Mobile', 2000), ('Laptop', 1500)]



### 6.2 Accumulators

Accumulators are **write-only shared variables** used for aggregating information (like counters or sums) across the workers.



They are commonly used for:

Counting bad records

Monitoring metrics

Why Use Accumulators?

When running tasks in parallel, Spark executes transformations independently.

They are write-only on executors (tasks can add to them) and read-only on the driver (you can read their final value after an action).

Accumulators allow each executor to add values to a shared variable safely — Spark aggregates them on the driver side.

#### Example:


In [ ]:
# NULL COUNT

from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("Accumulator Null Count Example") \
    .getOrCreate()

# Create an RDD with some None (null) values
rdd = spark.sparkContext.parallelize([10, None, 25, None, 40, 55, None])

# Define an accumulator
null_acc = spark.sparkContext.accumulator(0)

# Define a function to check for nulls
def count_nulls(x):
    global null_acc
    if x is None:
        null_acc.add(0)   # Increment accumulator
    return x

# Apply the function in a transformation
rdd.map(count_nulls).collect()   # Action triggers computation

# Print final accumulator value
print(f"Total Null Values: {null_acc.value}")


Total Null Values: -3


In [ ]:
# Use case: Data validation during ETL — counting malformed records.

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Accumulator Invalid Data").getOrCreate()

data = ["100", "200", , "300", "xyz", "400"]


rdd = spark.sparkContext.parallelize(data)

invalid_acc = spark.sparkContext.accumulator(0)

def validate_numeric(x):
    global invalid_acc
    try:
        return float(x)
    except ValueError:
        invalid_acc.add(1)
        return None

rdd.map(validate_numeric).collect()
print(f"Invalid records: {invalid_acc.value}")


SyntaxError: invalid syntax (3371883706.py, line 7)

In [ ]:
rdd = spark.sparkContext.textFile("customers.txt")

empty_line_acc = spark.sparkContext.accumulator(0)

def check_empty(line):
    global empty_line_acc
    if line.strip() == "":
        empty_line_acc.add(1)
    return line

rdd.map(check_empty).collect()
print(f"Empty lines: {empty_line_acc.value}")

Empty lines: 2
